# TP - Deep Learning avancé - attaques adverses

Le but de TP est d'implémenter l'attaque adverse d'un réseau CNN appris par ailleurs.

## Chargement des données

Pour ce TP, nous allons utiliser à nouveau utiliser le dataset MNIST (la drosophile du machine learing) pour faire un modèle de classification d'images. Nous allons donc, dans un premier temps, téléchargez le dataset et le préparer pour l'entraînement.

In [ ]:
# Chargement de MNIST

from torchvision import datasets, transforms

# Chargement des données
train_data = datasets.MNIST(...)
test_data = datasets.MNIST(...)


In [ ]:
#Créez vos dataloaders (attention à la taille des batchs !)

from torch.utils.data import DataLoader

## Création d'un réseau CNN

En reprenant le code du TP précédent, on crée un modèle CNN pour la classification des images de MNIST.

In [ ]:
# Definition du modele
import torch.nn as nn

class CNN(nn.Module):

Implémentez votre processus d'apprentissage sur 1 epoch avec un learning rate de 10e-3 (en utilisant l'optimiseur Adam) et une fonction de coût de type CrossEntropyLoss.

Vérifiez le taux de bonne classifcation en test de ce simple modèle.

## Attaque adverse

Instancier la fonction pgd_attack, qui étant donné un modèle, une image et son label, va produire une image corrompue telle que

$\max_e \mathcal{L}(x+\eta, y)$

tel que $x+\eta \in [0,1]^{n\times n}$ et $||\eta||_∞ \leqϵ$


Pour cela, on va faire une descente de gradient projeté à pas fixe (avec $\alpha = 0.01$ par défaut) et une nombre maximum d'itération.

On fera pour une nombre d'itération donné
- calcule du gradient $∇_x\mathcal{L}(x,y)$ et mise à jour $\eta ← \alpha * ∇_x\mathcal{L}(x,y)$. En pratique, on utilisera le signe du gradient.
- vérification de la contrainte $||\eta||_∞ \leq ϵ$ (on utilisera torch.clamp pour que $∀ i, -ϵ\leq \eta_i \leq ϵ$)
- image adversariale $ x'← x + \eta$
- projection de $x'$ sur l'espace des solution admissible (on utilisera à nouveau torch.clamp pour que $\forall (i,j), x'_{ij} \in [0, 1] $)

NB :
* pour notre cas, $\mathcal{L}$ sera l'entropie croisée, on cherchera une perburbation qui change le label (en pratique, le classifieur ira au moindre effort et essayera de "pousser" vers la deuxième classe la plus probable).
* on est sur une attaque non specifiée (on trompe le classifieur mais pas vers une classe particulière) mais on pourrait forcer le classifieur vers une classe définie

In [ ]:
# attaque PGD
def pgd_attack(model, images, labels, epsilon=0.3, alpha=0.01, num_iter=20):
    images = images.clone().detach().to(torch.float).requires_grad_(True)
    original_images = images.data

    for _ in range(num_iter):


        # PGD update step


    return images

In [ ]:
# Evaluation du modele sous attaque PGD
def evaluate_under_attack(model, test_loader, epsilon=0.3):
    model.eval()
    correct = 0
    total = 0
    for data, target in test_loader:
        adv_data = pgd_attack(model, data, target, epsilon=epsilon)
        output = model(adv_data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

    print(f'Accuracy sous attaque PGD (ε={epsilon}): {100 * correct / total:.2f}%')

In [ ]:
# evaluation du modele en pratique
evaluate_under_attack(model, test_loader, epsilon=0.3)

## Visualisation

- Prendre la première image de l'ensemble de test et l'afficher (en utilisant imshow de Matplotlib) et comparer la prédiction par le CCN de son label avec la vérité terrain.
- Générer une image adversarial à partir de cette image, l'afficher et comparer la prédiction du CNN sur cet exemple.

NB
* la norme $\ell_∞$ utilisée rend l'attaque peu discrète, une norme  $\ell_1$ ou $\ell_0$ serait plus furtive. On pourrait aussi considérer $\ell_2$.
* malgré l'attaque, l'exemple reste facilement identifiable pour un humain.

In [ ]:
import matplotlib.pyplot as plt



## pour aller plus loin
- tester la sensibilité de l'attaque aux différents paramètres ($\alpha$, $\epsilon$, nombre d'itérations... )
- instantier une autre attaque FGSM ou Carlini and Wagner (C&W)
- inclure cette attaque pour faire de l'adversarial training (on générera de nouveaux exemples adversariaux à chaque epoch)
- attaquer un réseau pré-appris (en téléchargeant les poids d'un CNN LeNet pour ImageNet).
- lire le papier "EXPLAINING AND HARNESSING
ADVERSARIAL EXAMPLES" (Goodfellow et al. ICLR 2015) https://arxiv.org/pdf/1412.6572
- lire le papier "Towards Deep Learning Models Resistant to Adversarial Attacks" (Madry et al. ICLR 2018) https://openreview.net/pdf?id=rJzIBfZAb